In [15]:
from collections import OrderedDict

def get_params_by_keywords(state_dict, keywords):
    """
    Filters a state_dict to include only parameters whose names contain any of the specified keywords.

    Args:
        state_dict (dict or OrderedDict): The model's state_dict.
        keywords (str or list of str): A keyword or a list of keywords to search for in parameter names.

    Returns:
        OrderedDict: A new dictionary containing only the matching parameters.
                    Using OrderedDict to preserve original parameter order.
    """
    if isinstance(keywords, str):
        keywords = [keywords]  # Convert single keyword to list for uniformity

    # Using a dictionary comprehension for conciseness
    filtered_params = OrderedDict({
        param_name: param_tensor
        for param_name, param_tensor in state_dict.items()
        if any([keyword in param_name for keyword in keywords])  # Include if any keyword matches
    })
    return filtered_params

def get_params_without_keywords(state_dict, keywords_to_exclude):
    """
    Filters a state_dict to include only parameters whose names
    do NOT contain any of the specified keywords.

    Args:
        state_dict (dict or OrderedDict): The model's state_dict.
        keywords_to_exclude (list of str): A list of keywords to exclude from parameter names.

    Returns:
        OrderedDict: A new dictionary containing only the parameters whose
                    names do not contain any of the specified keywords.
    """
    # Using a dictionary comprehension for conciseness
    filtered_params = OrderedDict({
        param_name: param_tensor
        for param_name, param_tensor in state_dict.items()
        if not any([keyword in param_name for keyword in keywords_to_exclude])  # Exclude if any keyword matches
    })
    return filtered_params


In [2]:
import torch

s12ep_path = '/data/all_checkpoints/MT_mistral7b_string_only_12ep_0415/epoch=11-step=40991.ckpt'
s12ep_ckpt = torch.load(s12ep_path, map_location='cpu', weights_only=False) 

qformer_path = '/data/all_checkpoints/MT_from-string_only_qformer-gine_tokengt_pretraining_1ep_0501/epoch=00-step=3415.ckpt'
qformer_ckpt = torch.load(qformer_path, map_location='cpu', weights_only=False)

In [8]:
s12ep_keys = list(s12ep_ckpt['state_dict'].keys())
qformer_keys = list(qformer_ckpt['state_dict'].keys())

In [11]:
s12ep_ckpt['state_dict']['blip2model.llm_model.base_model.model.model.embed_tokens.weight']

tensor([[-2.0336e-36,  3.3208e-37, -1.5517e-35,  ..., -4.9371e-36,
         -7.9934e-36, -5.9480e-36],
        [-4.3335e-03,  2.0218e-04, -5.5847e-03,  ...,  2.1935e-04,
         -7.7438e-04, -1.2207e-04],
        [-1.4420e-03,  8.1253e-04,  1.6880e-04,  ...,  1.2207e-03,
          2.2888e-04, -8.6784e-05],
        ...,
        [-1.8463e-03, -7.5989e-03,  9.3384e-03,  ..., -8.8501e-03,
         -1.2817e-02, -1.5625e-02],
        [ 3.3691e-02,  1.9043e-02, -2.9449e-03,  ..., -1.4648e-02,
         -5.6763e-03, -5.5847e-03],
        [-2.0020e-02, -1.9775e-02,  3.0396e-02,  ..., -1.0315e-02,
          4.1504e-03,  4.0283e-03]], dtype=torch.bfloat16)

In [20]:
filtered_keys = list(get_params_without_keywords(s12ep_ckpt['state_dict'], qformer_keys).keys())
[print(key) for key in filtered_keys if 'lora' not in key]

compensated_qformer_state_dict = qformer_ckpt['state_dict'].copy()
for key in filtered_keys:
    compensated_qformer_state_dict[key] = s12ep_ckpt['state_dict'][key]

In [21]:
get_params_without_keywords(compensated_qformer_state_dict, qformer_keys).keys()

odict_keys(['blip2model.llm_model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'blip2model.llm_model.base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'blip2model.llm_model.base_model.model.model

In [17]:
filtered_keys

['blip2model.llm_model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight',
 'blip2model.llm_model.base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight',
 'blip2model.llm_model.base_model.model.model.

In [24]:
qformer_ckpt['state_dict'] = compensated_qformer_state_dict
len(list(get_params_by_keywords(qformer_ckpt['state_dict'], ['lora']).keys()))

448

In [27]:
print(filtered_keys[0])
qformer_ckpt['state_dict'][filtered_keys[0]].mean()

blip2model.llm_model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight


tensor(-2.2247e-05)

In [28]:
# save qformer_ckpt
torch.save(qformer_ckpt, '/data/all_checkpoints/MT_from-string_only_qformer-gine_tokengt_pretraining_1ep_0501/epoch=00-step=3415_lora_compensated.ckpt')